# Experiment 032: seed-matched warm starts, 40 GB (Colab)

This session runs the four 0.5B and 3B arms. All four source gates finish before
any training starts. The old wsft endpoints stay unchanged and are used only as
references.


## 0. Compute units and mount Drive

Fill the two values before starting.


In [2]:
import glob, json, os, time
session_id = globals().get('session_id', time.strftime('%Y%m%d_%H%M%S'))
cu_before = 543.14
unit_rate = 5.3
print('session', session_id, '| cu_before =', cu_before if cu_before is not None else 'FILL ME')

from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/inm363'
RESULTS = os.path.join(ROOT, 'results')
OPEN_UNITS = os.path.join(RESULTS, 'open_cu_sessions')
UNITS_LOG = os.path.join(RESULTS, 'colab_units.jsonl')
os.makedirs(OPEN_UNITS, exist_ok=True)

old_sessions = [path for path in glob.glob(os.path.join(OPEN_UNITS, '*.json'))
                if not path.endswith(f'{session_id}.json')]
open_path = os.path.join(OPEN_UNITS, f'{session_id}.json')
if cu_before is None or unit_rate is None:
    print('fill cu_before and unit_rate, then rerun this cell')
else:
    for old_path in old_sessions:
        with open(old_path) as f:
            old = json.load(f)
        old['ts_end'] = time.strftime('%Y-%m-%d %H:%M')
        old['colab_cu_after'] = cu_before
        old['colab_cu_used'] = round(old['colab_cu_before'] - cu_before, 2)
        old['closed_on_resume'] = True
        with open(UNITS_LOG, 'a') as f:
            f.write(json.dumps(old) + '\n')
        os.remove(old_path)
        print('closed previous session:', old['session_id'])
    if not os.path.exists(open_path):
        record = {'session_id': session_id, 'ts_start': time.strftime('%Y-%m-%d %H:%M'),
                  'notebook': '032_colab_wsft_40gb', 'colab_cu_before': cu_before,
                  'unit_rate': unit_rate}
        with open(open_path, 'w') as f:
            json.dump(record, f, indent=2)
        print('saved open session:', session_id)


session 20260814_172706 | cu_before = 543.14
Mounted at /content/drive
closed previous session: 20260814_170421
saved open session: 20260814_172706


## 1. GPU check: A100 40GB


In [3]:
import subprocess, torch
gpu = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout
print(gpu.strip() or 'no gpu visible')
total_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'total vram: {total_gb:.1f} GiB')
assert 'A100' in gpu and 39 <= total_gb < 60, 'need the A100 40GB runtime'


GPU 0: NVIDIA A100-SXM4-40GB (UUID: GPU-e7617a0d-1443-b72a-6521-e4b7e5b7d36a)
total vram: 39.5 GiB


## 2. Clone or update the repo


In [4]:
import base64
import getpass
import glob
import os
from pathlib import Path
import subprocess

REPO_SLUG = 'hastov-lab/rl-grpo-or-sft-for-llms'
repo_url = f'https://github.com/{REPO_SLUG}.git'
repo_root = Path('/content') / REPO_SLUG.split('/')[-1]


def repo_git(*args):
    git = ['git']
    env = None
    try:
        from google.colab import userdata
    except ImportError:
        userdata = None

    # Private forks can use a read token from Colab Secrets.
    if userdata is not None and REPO_SLUG != 'hastov-lab/rl-grpo-or-sft-for-llms':
        try:
            token = userdata.get('GH_TOKEN')
        except Exception:
            token = None
        if isinstance(token, dict):
            token = token.get('value')
        if not isinstance(token, str) or not token.strip():
            token = getpass.getpass('GitHub token: ')
        if not token.strip():
            raise RuntimeError('This private fork requires a GitHub token with read access.')

        # Keep the token out of the command line and the saved remote URL.
        auth = base64.b64encode(f'x-access-token:{token.strip()}'.encode()).decode()
        env = dict(os.environ, REPO_GIT_AUTH=f'Authorization: Basic {auth}')
        git.append('--config-env=http.https://github.com/.extraHeader=REPO_GIT_AUTH')

    subprocess.run([*git, *args], env=env, check=True)


if repo_root.exists():
    if not (repo_root / '.git').exists():
        raise RuntimeError(f'Expected a Git checkout at {repo_root}.')
    origin = subprocess.check_output(
        ['git', '-C', str(repo_root), 'config', '--local', '--get', 'remote.origin.url'], text=True
    ).strip()
    if origin != repo_url:
        raise RuntimeError(f'The checkout at {repo_root} belongs to a different repository.')
    branch = subprocess.check_output(
        ['git', '-C', str(repo_root), 'branch', '--show-current'], text=True
    ).strip()
    if branch != 'main':
        raise RuntimeError(f'Expected the main branch at {repo_root}.')
    repo_git('-C', str(repo_root), 'pull', '--ff-only', 'origin', 'main')
else:
    repo_git('clone', '--branch', 'main', '--single-branch', '--depth', '1', repo_url, str(repo_root))

# Use the scripts from this checkout even if another experiment is open.
code_root = repo_root / '001_Code'
if not (code_root / 'scripts' / 'run.py').is_file():
    raise RuntimeError(f'Experiment scripts are missing from {code_root}.')
os.chdir(code_root)
print('working in:', os.getcwd())


working in: /content/INM363-IndividualProject-Dissertation/001_Code


## 3. Install dependencies


In [5]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'trl==1.7.1', 'transformers==5.5.4', 'peft==0.19.1',
                'accelerate==1.14.0', 'datasets==4.8.4', 'wandb', 'pyyaml'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-q', '-y', 'torchao'], check=True)
print('dependencies installed')


dependencies installed


## 4. W&B key

Grant access to `WANDB_API_KEY` in Colab secrets. The prompt is a
fallback when the W&B secret is unavailable. This key is only used for experiment tracking.


In [6]:
import getpass, os
wkey = None
try:
    from google.colab import userdata
    wkey = userdata.get('WANDB_API_KEY')
except Exception:
    wkey = None
if not wkey:
    wkey = getpass.getpass('W&B API key: ').strip()
if not wkey:
    raise RuntimeError('WANDB_API_KEY is required')
os.environ['WANDB_API_KEY'] = wkey
os.environ['WANDB_PROJECT'] = 'inm363-grpo'
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
with open('wandb_api_key.txt', 'w') as f:
    f.write(wkey.strip())
print('W&B key loaded')


W&B key loaded


## 5. Session setup


In [7]:
import os, shutil, sys
os.chdir(code_root)
sys.path.insert(0, os.getcwd())
from config import load_config
from scripts.s32_wsft import (ARMS_032, SCALES_032, arm_tag, config_path,
                              historical_tag, pipeline_config_path, pipeline_tag,
                              plain_tag)

PHASE = '40'
ARMS = list(ARMS_032[PHASE])
assert len(ARMS) == 4
assert cu_before is not None and unit_rate is not None, 'fill section 0 first'
free_gb = shutil.disk_usage('/content/drive/MyDrive').free / 1024**3
print(f'drive free: {free_gb:.0f} GB')
assert free_gb >= 40, 'need at least 40GB free on Drive'
print('arms:', ARMS)


drive free: 178 GB
arms: [('B05', 1), ('B05', 2), ('B3', 1), ('B3', 2)]


## 6. Wiring checks


In [8]:
import subprocess, sys

subprocess.run([
    sys.executable, '-m', 'pytest', '-q',
    'tests/test_032_wsft_sources.py',
    'tests/test_sft_chain.py',
    '-k', 'not test_032_notebooks_keep_the_card_group_gate_barrier',
], check=True)

for scale, seed in ARMS:
    subprocess.run([sys.executable, 'scripts/run.py', '--config', config_path(scale, seed)],
                   check=True)
print('wiring passed')


wiring passed


## 7. Structural preflight


In [9]:
import glob, hashlib, json, os, subprocess, sys
from checkpoint import config_hash
from obs import _git_info, _versions
from scripts.s12_eval import (ADAPTER_FILES, atomic_write_json, config_hash_status,
                              read_jsonl, sha256)
from scripts.s32_wsft import (append_failed_gate_contact_correction,
                              prepopulate_scale_base, verify_source_lineage)
from train.model_utils import resolve_dtype

D = '/content/drive/MyDrive/inm363'
append_failed_gate_contact_correction(os.path.join(D, 'results', 'compute_ledger.jsonl'))
current_versions = _versions()
scientific_paths = ['config.py', 'checkpoint.py', 'enums.py', 'file_utils.py', 'obs.py',
                    'scripts/run.py', 'scripts/s12_eval.py', 'scripts/s28_compose.py',
                    'scripts/s32_wsft.py',
                    'scripts/s32_start_probe.py', 'scripts/s32_sft_eval.py']
for folder in ('data', 'eval', 'train'):
    scientific_paths.extend(os.path.relpath(path)
                            for path in glob.glob(f'{folder}/**/*.py', recursive=True))
scientific_paths = sorted(set(scientific_paths))
execution = {
    'scientific_file_hashes': {path: sha256(path) for path in scientific_paths},
    'config_hashes': {config_path(scale, seed): config_hash(load_config(config_path(scale, seed)))
                      for group in ARMS_032.values() for scale, seed in group},
}
execution_path = os.path.join(D, 'results', '032_execution_binding_v6.json')
if os.path.exists(execution_path):
    assert json.load(open(execution_path)) == execution, '032 scientific code changed between sessions'
else:
    atomic_write_json(execution_path, execution)

preflight = {'version': 6, 'phase': PHASE, 'execution_sha256': sha256(execution_path),
             'historical': {}, 'arms': {}}
for scale in sorted({scale for scale, _ in ARMS}):
    cfg = load_config(config_path(scale, 1))
    destination = os.path.join(D, 'results', 'base_evals',
                               f'032_wsft_COLAB_FULL_{scale}')
    prepopulate_scale_base(D, scale, destination)

    for hist_seed in (0, 1, 2):
        hist_tag = historical_tag(scale, hist_seed)
        hist = os.path.join(D, 'results', hist_tag)
        names = ('sft_eval.json', 'sft_preds.jsonl',
                 'sft_logprob.json', 'sft_logprob_preds.jsonl')
        for name in names:
            assert os.path.exists(os.path.join(hist, name)), f'{hist}: missing {name}'
        assert json.load(open(os.path.join(hist, 'sft_eval.json')))['n_items'] == 1884
        assert json.load(open(os.path.join(hist, 'sft_logprob.json')))['n_items'] == 1884
        assert len(read_jsonl(os.path.join(hist, 'sft_preds.jsonl'))) == 1884
        assert len(read_jsonl(os.path.join(hist, 'sft_logprob_preds.jsonl'))) == 1884
        preflight['historical'][hist_tag] = {
            name: sha256(os.path.join(hist, name)) for name in names}

for scale, seed in ARMS:
    cfg_path = config_path(scale, seed)
    cfg = load_config(cfg_path)
    tag = arm_tag(scale, seed)
    assert cfg.train.seed == seed and cfg.train.max_steps == 123
    assert cfg.exec.allow_oom_fallback is False
    assert cfg.log.wandb_project == 'inm363-grpo'

    parent_tag = pipeline_tag(scale, seed)
    parent_result = os.path.join(D, 'results', parent_tag)
    parent_manifest_path = os.path.join(parent_result, 'manifest.json')
    parent = json.load(open(parent_manifest_path))
    for key in ('torch', 'trl', 'transformers', 'peft', 'accelerate',
                'cais/mmlu@', 'openai/gsm8k@'):
        expected = parent.get('versions', {}).get(key)
        if expected is not None:
            assert current_versions.get(key) == expected, f'{tag}: {key} differs from {parent_tag}'
    assert resolve_dtype(cfg) == parent['resolved_dtype'], f'{tag}: dtype differs from {parent_tag}'

    source_adapter = os.path.join(D, cfg.train.sft_from_adapter)
    for name in ADAPTER_FILES:
        assert os.path.exists(os.path.join(source_adapter, name)), f'{tag}: source missing {name}'
    archived_preds = os.path.join(parent_result, 'sft_logprob_preds.jsonl')
    assert os.path.exists(archived_preds), f'{tag}: archived source predictions missing'
    assert len(read_jsonl(archived_preds)) == 1884
    source_cfg = load_config(pipeline_config_path(scale, seed))
    stored_hash = open(os.path.join(source_adapter, 'config_hash.txt')).read().strip()
    hash_status = config_hash_status(source_cfg, parent, stored_hash)
    assert hash_status != 'mismatch', f'{tag}: source adapter config hash differs'

    result_dir = os.path.join(D, 'results', tag)
    adapter_dir = os.path.join(D, 'checkpoints', tag, 'sft')
    binding = verify_source_lineage(cfg, result_dir, adapter_dir, parent_manifest_path)
    preflight['arms'][tag] = {
        'config': cfg_path,
        'config_sha256': sha256(cfg_path),
        'parent_manifest_sha256': sha256(parent_manifest_path),
        'source_config_hash_status': hash_status,
        'archived_preds_sha256': sha256(archived_preds),
        'source_lineage': binding,
    }

preflight_path = os.path.join(D, 'results', f'032_preflight_{PHASE}_v6.json')
if os.path.exists(preflight_path):
    assert json.load(open(preflight_path)) == preflight, '032 preflight changed on rerun'
else:
    atomic_write_json(preflight_path, preflight)
print('structural preflight passed for', PHASE)


  base evals already prepopulated and verified
  base evals already prepopulated and verified
structural preflight passed for 40


## 8. Build the data splits


In [10]:
import subprocess, sys
subprocess.run([sys.executable, 'scripts/build_data.py'], check=True)


CompletedProcess(args=['/usr/bin/python3', 'scripts/build_data.py'], returncode=0)

## 9. Source identity gates


In [11]:
import json, os, shutil, subprocess, sys, time
from obs import append_ledger, gpu_snapshot
from scripts.s32_wsft import (GATE_FILES, commit_source_gate, source_gate_state,
                              verify_source_lineage)

LEDGER = os.path.join(D, 'results', 'compute_ledger.jsonl')
assert os.path.exists(preflight_path), 'run structural preflight first'
assert json.load(open(preflight_path)) == preflight, '032 v6 preflight changed'
for scale, seed in ARMS:
    cfg_path = config_path(scale, seed)
    cfg = load_config(cfg_path)
    tag = arm_tag(scale, seed)
    result_dir = os.path.join(D, 'results', tag)
    adapter_dir = os.path.join(D, 'checkpoints', tag, 'sft')
    parent_manifest = os.path.join(D, 'results', pipeline_tag(scale, seed), 'manifest.json')
    binding = verify_source_lineage(cfg, result_dir, adapter_dir, parent_manifest)
    archived_preds = os.path.join(D, 'results', pipeline_tag(scale, seed),
                                  'sft_logprob_preds.jsonl')
    if source_gate_state(result_dir, binding, archived_preds) == 'committed':
        print(tag, 'source gate already committed')
        continue

    scratch = f'/content/s32_gate_{scale}_seed{seed}'
    shutil.rmtree(scratch, ignore_errors=True)
    started = time.time()
    evaluated_items = 0
    committed = False
    try:
        process = subprocess.run([sys.executable, '-u', 'scripts/s32_start_probe.py',
                                  '--config', cfg_path, '--adapter', binding['adapter_dir'],
                                  '--out', scratch])
        if process.returncode != 0:
            raise RuntimeError(f'{tag}: source gate subprocess failed')
        commit_source_gate(result_dir, scratch, binding, archived_preds,
                           extra={'config': cfg_path, 'seed': seed, 'scale': scale,
                                  'gpu': gpu_snapshot()})
        committed = True
    finally:
        preds_path = os.path.join(scratch, GATE_FILES[1])
        if os.path.exists(preds_path):
            with open(preds_path) as f:
                evaluated_items = sum(bool(line.strip()) for line in f)
        append_ledger(LEDGER, {'run_tag': tag, 'condition': 'source_adapter_identity_gate',
                               'timings_sec': {'source_gate': round(time.time() - started, 1)},
                               'train_sec': 0, 'train_steps': 0,
                               'eval_items': {'source_gate': evaluated_items or None},
                               'evaluated': evaluated_items > 0, 'committed': committed,
                               'completed': committed, 'gpu': gpu_snapshot()})
    shutil.rmtree(scratch, ignore_errors=True)

for scale, seed in ARMS:
    cfg = load_config(config_path(scale, seed))
    tag = arm_tag(scale, seed)
    result_dir = os.path.join(D, 'results', tag)
    adapter_dir = os.path.join(D, 'checkpoints', tag, 'sft')
    parent_manifest = os.path.join(D, 'results', pipeline_tag(scale, seed), 'manifest.json')
    binding = verify_source_lineage(cfg, result_dir, adapter_dir, parent_manifest)
    archived_preds = os.path.join(D, 'results', pipeline_tag(scale, seed),
                                  'sft_logprob_preds.jsonl')
    assert source_gate_state(result_dir, binding, archived_preds) == 'committed'
print('all four source gates committed before training')


  source gate committed: {'agreement': 1.0, 'acc_new': 0.562102, 'acc_archived': 0.562102}
  source gate committed: {'agreement': 1.0, 'acc_new': 0.555732, 'acc_archived': 0.555732}
  source gate committed: {'agreement': 1.0, 'acc_new': 0.742569, 'acc_archived': 0.742569}
  source gate committed: {'agreement': 1.0, 'acc_new': 0.731423, 'acc_archived': 0.731423}
all four source gates committed before training


## 10. Train and evaluate the four arms


In [12]:
import json, os, shutil, subprocess, sys, time, torch
from obs import append_ledger, gpu_snapshot
from scripts.s32_wsft import (GATE_MANIFEST, arm_state, commit_sft_endpoint, endpoint_state,
                              source_gate_state, training_facts, verify_source_lineage,
                              write_arm_complete)

assert os.path.exists(preflight_path), 'run structural preflight first'
assert json.load(open(preflight_path)) == preflight, '032 v6 preflight changed'
# keep the card-group gate barrier inside this cell as well
gate_inputs = {}
for scale, seed in ARMS:
    cfg = load_config(config_path(scale, seed))
    tag = arm_tag(scale, seed)
    result_dir = os.path.join(D, 'results', tag)
    adapter_dir = os.path.join(D, 'checkpoints', tag, 'sft')
    parent_manifest = os.path.join(D, 'results', pipeline_tag(scale, seed), 'manifest.json')
    binding = verify_source_lineage(cfg, result_dir, adapter_dir, parent_manifest)
    archived_preds = os.path.join(D, 'results', pipeline_tag(scale, seed),
                                  'sft_logprob_preds.jsonl')
    assert source_gate_state(result_dir, binding, archived_preds) == 'committed'
    gate_inputs[(scale, seed)] = (cfg, result_dir, adapter_dir, binding, archived_preds)

for scale, seed in ARMS:
    cfg, result_dir, adapter_dir, binding, archived_preds = gate_inputs[(scale, seed)]
    tag = arm_tag(scale, seed)
    if arm_state(cfg, result_dir, adapter_dir, binding, archived_preds) == 'committed':
        print(tag, 'complete, skipped')
        continue

    final_adapter = os.path.join(adapter_dir, 'adapter_model.safetensors')
    if os.path.exists(final_adapter):
        required = ('sft_trainlog.json', 'sft_logprob.json', 'sft_logprob_preds.jsonl',
                    'manifest.json', 'summary.json')
        if all(os.path.exists(os.path.join(result_dir, name)) for name in required):
            training_facts(cfg, result_dir, adapter_dir, binding)
            trained = True
        else:
            trained = False
    else:
        trained = False

    if not trained:
        free_gib = torch.cuda.mem_get_info()[0] / 1024**3
        need_gib = SCALES_032[scale]['vram_gib']
        assert free_gib >= need_gib, f'{tag}: only {free_gib:.1f} GiB free, need {need_gib}'
        process = subprocess.run([sys.executable, '-u', 'scripts/run.py',
                                  '--config', config_path(scale, seed), '--stage', 'all'])
        if process.returncode != 0:
            raise RuntimeError(f'{tag}: SFT runner failed with exit code {process.returncode}')
    training_facts(cfg, result_dir, adapter_dir, binding)

    if endpoint_state(result_dir, adapter_dir) != 'committed':
        scratch = f'/content/s32_eval_{scale}_seed{seed}'
        shutil.rmtree(scratch, ignore_errors=True)
        started = time.time()
        completed = False
        try:
            process = subprocess.run([sys.executable, '-u', 'scripts/s32_sft_eval.py',
                                      '--config', config_path(scale, seed),
                                      '--adapter', adapter_dir, '--out', scratch])
            if process.returncode != 0:
                raise RuntimeError(f'{tag}: endpoint evaluation failed')
            commit_sft_endpoint(
                result_dir, scratch, adapter_dir,
                extra={'config': config_path(scale, seed), 'seed': seed, 'scale': scale,
                       'source_adapter_gate_manifest_sha256': sha256(os.path.join(
                           result_dir, GATE_MANIFEST)),
                       'start_binding': binding, 'gpu': gpu_snapshot()})
            completed = True
        finally:
            append_ledger(LEDGER, {'run_tag': tag, 'condition': 'SFT_ONLY_gen_eval',
                                   'timings_sec': {'sft_eval': round(time.time() - started, 1)},
                                   'train_sec': 0, 'train_steps': 0,
                                   'eval_items': {'sft': 1884 if completed else None},
                                   'completed': completed, 'gpu': gpu_snapshot()})
        shutil.rmtree(scratch, ignore_errors=True)

    write_arm_complete(cfg, result_dir, adapter_dir, binding, archived_preds)
    print(tag, 'complete')


  SFT endpoint committed
032_wsft_COLAB_FULL_B05_sft_only_seed1 complete
  SFT endpoint committed
032_wsft_COLAB_FULL_B05_sft_only_seed2 complete
  SFT endpoint committed
032_wsft_COLAB_FULL_B3_sft_only_seed1 complete
  SFT endpoint committed
032_wsft_COLAB_FULL_B3_sft_only_seed2 complete


## 11. Peek at the results


In [13]:
import json, os
for scale, seed in ARMS:
    tag = arm_tag(scale, seed)
    result_dir = os.path.join(D, 'results', tag)
    endpoint = os.path.join(result_dir, 'sft_eval.json')
    if not os.path.exists(endpoint):
        print(tag, 'not complete')
        continue
    metrics = json.load(open(endpoint))
    print(tag, metrics['per_tier_acc'])


032_wsft_COLAB_FULL_B05_sft_only_seed1 {'easy': 0.6530147895335608, 'medium': 0.46469622331691296, 'hard': 0.34595959595959597}
032_wsft_COLAB_FULL_B05_sft_only_seed2 {'easy': 0.6564277588168373, 'medium': 0.4794745484400657, 'hard': 0.3409090909090909}
032_wsft_COLAB_FULL_B3_sft_only_seed1 {'easy': 0.8543799772468714, 'medium': 0.7274220032840722, 'hard': 0.46464646464646464}
032_wsft_COLAB_FULL_B3_sft_only_seed2 {'easy': 0.856655290102389, 'medium': 0.7389162561576355, 'hard': 0.48737373737373735}


## 12. Close the units record


In [ ]:
import json, os, time
cu_after = 531.3 
if cu_after is None:
    print('fill cu_after, then rerun this cell')
elif not os.path.exists(open_path):
    print('no open record for this session')
else:
    with open(open_path) as f:
        record = json.load(f)
    record['ts_end'] = time.strftime('%Y-%m-%d %H:%M')
    record['colab_cu_after'] = cu_after
    record['colab_cu_used'] = round(record['colab_cu_before'] - cu_after, 2)
    record['closed_on_resume'] = False
    with open(UNITS_LOG, 'a') as f:
        f.write(json.dumps(record) + '\n')
    os.remove(open_path)
    print('logged:', record)


fill cu_after, then rerun this cell
